# Live GEE pipeline — recompute the numbers yourself

This notebook calls Google Earth Engine directly to recompute Mask A, Mask B and the EUDR-risk conversion mask for an AOI from the registry in `src/aois.py`. You need:

- A GEE account with a **registered Cloud project** (see [Earth Engine sign-up](https://earthengine.google.com/) and [project registration](https://developers.google.com/earth-engine/cloud/projects))
- `earthengine-api` and `geemap` installed (uncomment the pip line below)
- Internet access

If you do not have GEE access, the published snapshot in `data/runs/2026-06-08/` is sufficient to reproduce every number cited in the companion article. Use `notebooks/00_reproduce_article_numbers.ipynb` instead.

## How to configure the GEE project ID

The setup cell reads the project ID from the environment variable `EE_PROJECT`. Set it before launching Jupyter:

```bash
export EE_PROJECT='your-registered-project-id'
jupyter notebook
```

Alternatively override it inline by setting `EE_PROJECT = '...'` directly in the setup cell.

In [ ]:
# %pip install -q earthengine-api geemap

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import ee

EE_PROJECT = os.environ.get('EE_PROJECT')
if not EE_PROJECT:
    raise RuntimeError(
        'Set the EE_PROJECT environment variable to your registered Earth '
        'Engine Cloud project ID before running this notebook (see the '
        'markdown cell above for details).'
    )

ee.Initialize(project=EE_PROJECT)
print(f'Earth Engine initialised on project {EE_PROJECT!r}')

from src.aois import ee_geometry, get_aoi, AOIS
from src.gee import conversion_mask, recent_loss_mask, plantation_probability_image

## Pick an AOI

The two 33 × 33 km town-centred AOIs used in the companion article are `civ_soubre_33km` and `gha_sefwi_wiawso_33km`. Larger administrative sub-regions are also available; see `src.aois.AOIS`.

In [ ]:
aoi_id = 'civ_soubre_33km'
aoi = get_aoi(aoi_id)
geom = ee_geometry(aoi_id)
print(aoi)
print('Available AOI IDs:', list(AOIS))

## Compute the masks and the EUDR-risk share

In [ ]:
mask_a = recent_loss_mask(geom)
prob = plantation_probability_image(geom)
risk = conversion_mask(geom)

pixel_area_ha = ee.Image.pixelArea().divide(10000)

def sum_ha(image):
    res = image.multiply(pixel_area_ha).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=30,
        maxPixels=1e10, bestEffort=True,
    ).getInfo()
    band = next(iter(res))
    return float(res[band])

plant_ha = sum_ha(prob.gte(0.5))
risk_ha = sum_ha(risk)
risk_share_pct = 100 * risk_ha / plant_ha if plant_ha else float('nan')

print(f'AOI: {aoi_id}')
print(f'Plantation area (FDP prob >= 0.5): {plant_ha:,.0f} ha')
print(f'EUDR-risk area:                    {risk_ha:,.0f} ha')
print(f'EUDR-risk share:                   {risk_share_pct:.2f} %')

## Sanity check against the article snapshot

For the two article AOIs the cell below compares the live result to the published snapshot. A larger drift than the tolerance signals that the FDP layer or the Hansen vintage on GEE has rolled forward.

In [ ]:
EXPECTED = {
    'civ_soubre_33km':       2.74,
    'gha_sefwi_wiawso_33km': 6.05,
}
TOL = 0.20  # percentage points

if aoi_id in EXPECTED:
    delta = risk_share_pct - EXPECTED[aoi_id]
    print(f'Live result:      {risk_share_pct:.2f} %')
    print(f'Article snapshot: {EXPECTED[aoi_id]:.2f} %')
    print(f'Delta:            {delta:+.2f} pp')
    if abs(delta) > TOL:
        print(f'WARNING: live result drifted more than {TOL} pp from the article snapshot.')
    else:
        print('Within tolerance — the article snapshot is still defendable.')
else:
    print(f'No published article reference for AOI {aoi_id!r}.')

## Reproduction caveats

- The result will shift slightly between runs as GEE updates the underlying assets. The article's quoted numbers (Soubré 2.74 %, Sefwi-Wiawso 6.05 %) come from the 2026-06-08 run.
- The pixel-area computation above is approximate at AOI scale. For audit-grade area accounting follow Olofsson et al. 2014 *RSE* good practice (stratified random sample + Cohen's kappa).
- For the full licence picture of every input layer (and the commercial-vs-non-commercial split) see the `Licence` section of the repository `README.md` and the `data/runs/2026-06-08/README.md` licence-pitfall block. Derived numbers from the FDP cocoa layer carry FDP's non-commercial restriction; the methodology and the code in this repo do not.